In [8]:
import os
import uuid
import pickle
import pandas as pd
import mlflow
from sklearn.metrics import root_mean_squared_error
from mlflow.tracking import MlflowClient
import xgboost

In [7]:
RUN_ID = "1733bcf5fc484bdfb50461b0182158fd"
os.environ["AWS_PROFILE"] = "user1"

# Downloading dictvectorizer from AWS
TRACKING_SERVER_HOST = "ec2-18-159-149-230.eu-central-1.compute.amazonaws.com"
mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:5000")
client = MlflowClient(f"http://{TRACKING_SERVER_HOST}:5000")

mlflow.set_experiment("nyc-taxi-experiment")

path = client.download_artifacts(run_id=RUN_ID, path='preprocessor', dst_path='.')

print(f"Downloading dictvectorizer to {path}")

with open('preprocessor/preprocessor.b', "rb") as f_out:
    dv = pickle.load(f_out)


year = 2021
month = 2
taxi_type = 'green'

input_file = f'https://s3.amazonaws.com/nyc-tlc/trip+data/{taxi_type}_tripdata_{year:04d}-{month:02d}.parquet'
output_file = f'output/{taxi_type}/{year:04d}-{month:02d}.parquet'

In [5]:
def generate_uuids(n):
    ride_ids = []
    for i in range(n):
        ride_ids.append(str(uuid.uuid4()))
    return ride_ids

def read_dataframe(filename: str):
    df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.dt.total_seconds() / 60
    df = df[(df.duration >= 1) & (df.duration <= 60)]
    
    df['ride_id'] = generate_uuids(len(df))

    return df


def prepare_dictionaries(df: pd.DataFrame):
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']

    categorical = ['PU_DO']
    numerical = ['trip_distance']
    dicts = df[categorical + numerical].to_dict(orient='records')
    dicts = dv.transform(dicts)
    return dicts

In [9]:
def load_model(run_id):
    logged_model = f"runs:/{run_id}/models"
    model = mlflow.xgboost.load_model(logged_model)
    return model


def apply_model(input_file, run_id, output_file):

    df = read_dataframe(input_file)
    dicts = prepare_dictionaries(df)
    X = xgboost.DMatrix(dicts)
    
    model = load_model(run_id)
    y_pred = model.predict(X)

    df_result = pd.DataFrame()
    df_result['ride_id'] = df['ride_id']
    df_result['lpep_pickup_datetime'] = df['lpep_pickup_datetime']
    df_result['PULocationID'] = df['PULocationID']
    df_result['DOLocationID'] = df['DOLocationID']
    df_result['actual_duration'] = df['duration']
    df_result['predicted_duration'] = y_pred
    df_result['diff'] = df_result['actual_duration'] - df_result['predicted_duration']
    df_result['model_version'] = run_id
    
    df_result.to_parquet(output_file, index=False)

In [ ]:
apply_model(input_file=input_file, run_id=RUN_ID, output_file=output_file)